## Settings

In [22]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [ ]:
## libraries
import sys
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import load_processed_data
from src.estimators.factories import load_estimators
from src.evaluators.resampling import (
    logo_cross_valid,
    kfold_cross_valid
)
from src.evaluators.transfer import (
    compile_domain_transfer,
    results_domain_transfer
)

## constants
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z, 
    TARGET
)

## Initialization

In [24]:
## reproducibility
N_REPEATS = 30
RANDOM_STATE = 42

## load data and models
data = load_processed_data()
models = load_estimators(random_state = RANDOM_STATE)

## view data shape
n_obs, n_feat = data.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")

## view model surface
n_mods = len(models)
print(f"Learned Models: {n_mods} estimators")

Original Data: 32 features, 25 observations
Learned Models: 9 estimators


## Training

In [25]:
## leave-one-group-out cross validation (domain)
if "results_dict_domain" not in globals():
    results_dict_domain = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, _ = logo_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            group = "domain",  ## fold on domain
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontier["model"] = name
        results_dict_domain[name] = frontier


In [26]:
## leave-one-group-out cross validation (discipline)
if "results_dict_disc" not in globals():
    results_dict_disc = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, _ = logo_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            group = "discipline",  ## fold on discipline
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontier["model"] = name
        results_dict_disc[name] = frontier


In [27]:
## 5-fold cross validation
if "results_dict_5fold" not in globals():
    results_dict_5fold = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, _ = kfold_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            n_splits = 5,  ## fold on random 80-20 split
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontier["model"] = name
        results_dict_5fold[name] = frontier


In [28]:
## 10-fold cross validation
if "results_dict_10fold" not in globals():
    results_dict_10fold = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, _ = kfold_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            n_splits = 10,  ## fold on random 90-10 split
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontier["model"] = name
        results_dict_10fold[name] = frontier


## Post-Processing

In [32]:
## compile domain-transfer results
results_data_domain = compile_domain_transfer(results = results_dict_domain)
results_data_disc = compile_domain_transfer(results = results_dict_disc)
results_data_5fold = compile_domain_transfer(results = results_dict_5fold)
results_data_10fold = compile_domain_transfer(results = results_dict_10fold)

## Domain Transfer Tests
Four resampling regimes evaluate held-out-group transferability against random-split baselines. Leave-one-group-out (LOGO) cross-validation by domain and discipline assesses generalization to unseen groups; random 5-fold and 10-fold cross-validation establish reference baselines.

- **Domain (LOGO)**: Held-out across 5 domains; group splits fixed, learner initializations vary across repeats.
- **Discipline (LOGO)**: Held-out across 25 disciplines; group splits fixed, learner initializations vary across repeats.
- **Random (5-Fold)**: Repeated shuffled 80–20 splits.
- **Random (10-Fold)**: Repeated shuffled 90–10 splits.

### Reporting Convention
Each regime contributes one value per estimator, averaged across groups or folds and repeats. The summary table reports the median across estimators. `EI` is shown as median `[Q1–Q3]`; `VR`, `MV`, and `MS` report medians, with `MV` and `MS` on the log-target scale. Aggregations are equally weighted across groups, folds, repeats, and estimators.

In [33]:
## compile transfer bounds and baseline benchmarks
results = results_domain_transfer(
    {"Domain (LOGO)": results_data_domain, "Discipline (LOGO)": results_data_disc},
    {"Random (5-Fold)": results_data_5fold, "Random (10-Fold)": results_data_10fold},
    keys = ("Transfer", "Standard"),
    indicies = ("Evaluation", "Method"),
    n_repeats = N_REPEATS,
    random_state = RANDOM_STATE,
    decimals = 2
)

display(results)

Cross-Validation: 9 models, 30 repeats (seeds 42-71)
Across-model aggregation: median of model-level means
Resampling: LOGO splits are fixed across repeats; random k-fold splits are reshuffled across repeats
Weighting: groups, folds, repeats, and models are equally weighted; results are not observation-weighted


Transfer `EI` is 0.79 (Domain LOGO) and 0.76 (Discipline LOGO), comparable to the random-split baselines (0.79–0.80). Violation rates differ across regimes (0.044–0.080) but remain low throughout, suggesting that estimator performance under strict held-out-group evaluation does not depart materially from random-partition baselines.

## Visualization